In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output, FileLink
import time
from datetime import datetime
import random
import csv
import io

def log(message):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {message}")

# Çıktıları göstereceğimiz ana alan
output_area = widgets.Output()
display(output_area)

# Task durumlarını göstermek için bir HTML widget'ı
status_label = widgets.HTML(value="<b style='color: black;'>DAG Bekliyor</b>")
display(status_label)

# İlerleme çubuğu
progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=3,  # Toplam 3 görev var: Extract, Transform, Report
    description='İlerleme:',
    bar_style='info',
    orientation='horizontal'
)
display(progress_bar)

# Dropdown menü için ayrı bir çıktı alanı
dropdown_output_area = widgets.Output()
display(dropdown_output_area)

# --- Task Sonuçları Tablosu ---
# GridspecLayout oluştur
task_results_table = widgets.GridspecLayout(4, 3, width='500px') # 4 satır (header + 3 task), 3 sütun

# Header oluştur
task_results_table[0, 0] = widgets.HTML("<b>Task Adı</b>")
task_results_table[0, 1] = widgets.HTML("<b>Durum</b>")
task_results_table[0, 2] = widgets.HTML("<b>Süre (s)</b>")

# Task satırlarını başlangıçta boş veya bekliyor olarak ayarla
task_results_table[1, 0] = widgets.HTML("Extract")
task_results_table[1, 1] = widgets.HTML("<span style='color: grey;'>Bekliyor</span>")
task_results_table[1, 2] = widgets.HTML("-")

task_results_table[2, 0] = widgets.HTML("Transform")
task_results_table[2, 1] = widgets.HTML("<span style='color: grey;'>Bekliyor</span>") # Fixed: Initialize with HTML widget
task_results_table[2, 2] = widgets.HTML("-") # Fixed: Initialize with HTML widget

task_results_table[3, 0] = widgets.HTML("Report")
task_results_table[3, 1] = widgets.HTML("<span style='color: grey;'>Bekliyor</span>") # Fixed: Initialize with HTML widget
task_results_table[3, 2] = widgets.HTML("-") # Fixed: Initialize with HTML widget

display(task_results_table)

def run_task_ipy(task_name, row_index):
    start_time = datetime.now()

    with output_area:
        status_label.value = f"<b style='color: orange;'>{task_name} RUNNING</b>"
        task_results_table[row_index, 1].value = "<span style='color: orange;'>Çalışıyor</span>"
        task_results_table[row_index, 2].value = "Çalışıyor..."

        log(f"{task_name} task başladı")
        log("Task instance oluşturuldu")

        records = random.randint(500, 5000)

        if task_name == "Extract":
            log("Kaynak sistem bağlantısı kuruluyor...")
            time.sleep(1)

            log("Veritabanı bağlantısı başarılı")
            time.sleep(1)

            log(f"{records} kayıt çekildi")

        elif task_name == "Transform":
            log("Veri doğrulama başladı")
            time.sleep(1)

            log("Eksik değer kontrolü tamamlandı")
            time.sleep(1)

            log("Feature engineering uygulandı")
            time.sleep(1)

            log(f"{records} kayıt dönüştürüldü")

        elif task_name == "Report":
            log("Rapor oluşturuluyor")
            time.sleep(1)

            log("Grafikler hazırlanıyor")
            time.sleep(1)

            log("PDF çıktısı oluşturuldu")
            time.sleep(1)

        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()

        log(f"{task_name} SUCCESS")
        log(f"Süre: {duration:.2f} saniye")

    status_label.value = f"<b style='color: green;'>{task_name} COMPLETED</b>"
    task_results_table[row_index, 1].value = "<span style='color: green;'>Tamamlandı</span>"
    task_results_table[row_index, 2].value = f"{duration:.2f}"
    progress_bar.value += 1

def download_results_clicked(b):
    output = io.StringIO()
    csv_writer = csv.writer(output)

    # CSV Header
    header_row = [
        task_results_table[0, 0].value.replace('<b>', '').replace('</b>', ''),
        task_results_table[0, 1].value.replace('<b>', '').replace('</b>', ''),
        task_results_table[0, 2].value.replace('<b>', '').replace('</b>', '')
    ]
    csv_writer.writerow(header_row)

    # CSV Data Rows
    for i in range(1, 4):
        task_name = task_results_table[i, 0].value
        status = task_results_table[i, 1].value.replace("<span style='color: green;'>", '').replace("</span>", '')
        duration = task_results_table[i, 2].value
        csv_writer.writerow([task_name, status, duration])

    file_name = f"dag_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    with open(file_name, 'w', newline='') as f:
        f.write(output.getvalue())

    with output_area:
        print("\nSonuç dosyası hazır:")
        display(FileLink(file_name))

def run_dag_ipy(b):
    with output_area:
        clear_output(wait=True)
        print("DAG başlatıldı!")

    status_label.value = "<b style='color: blue;'>DAG başlatıldı</b>"
    progress_bar.value = 0 # Yeni DAG çalıştırmasında ilerleme çubuğunu sıfırla
    download_button.disabled = True # DAG başladığında indirme düğmesini devre dışı bırak
    save_plot_button.disabled = True # Grafiği kaydet düğmesini de devre dışı bırak

    # Tabloyu sıfırla
    task_results_table[1, 1].value = "<span style='color: grey;'>Bekliyor</span>"
    task_results_table[1, 2].value = "-"
    task_results_table[2, 1].value = "<span style='color: grey;'>Bekliyor</span>"
    task_results_table[2, 2].value = "-"
    task_results_table[3, 1].value = "<span style='color: grey;'>Bekliyor</span>"
    task_results_table[3, 2].value = "-"

    run_task_ipy("Extract", 1)
    run_task_ipy("Transform", 2)
    run_task_ipy("Report", 3)

    with output_area:
        clear_output(wait=True)
        status_label.value = "<b style='color: green;'>DAG Tamamlandı</b>"
        print("DAG Tamamlandı!")

    download_button.disabled = False # DAG tamamlandığında indirme düğmesini etkinleştir
    save_plot_button.disabled = False # Grafiği kaydet düğmesini de etkinleştir


# Bir düğme oluşturun
button = widgets.Button(description="DAG Çalıştır")

# Düğmeye tıklanıldığında run_dag_ipy fonksiyonunu çağırın
button.on_click(run_dag_ipy)

# Düğmeyi görüntüle
display(button)

# İndirme düğmesi oluşturun
download_button = widgets.Button(description="Sonuçları İndir", disabled=True)
download_button.on_click(download_results_clicked)
display(download_button)

# Kaydetme düğmesi oluşturun
save_plot_button = widgets.Button(description="Grafiği PNG Olarak Kaydet", disabled=True)
save_plot_button.on_click(lambda b: create_summary_plot(save_png=True))
display(save_plot_button)

# Dropdown menü oluşturun
dropdown = widgets.Dropdown(
    options=['DAG Adımı Seçin', 'Extract', 'Transform', 'Report'],
    value='DAG Adımı Seçin',
    description='Adım Detayı:',
    disabled=False,
)
display(dropdown)

def on_dropdown_change(change):
    with dropdown_output_area:
        clear_output(wait=True)
        selected_step = change.new
        if selected_step == 'Extract':
            print("Extract Adımı: Ham veriler veri kaynağından çekildi ve temel temizlemeler yapıldı. Yaklaşık 1000 satır veri.")
            print("Veri önizlemesi: İlk 5 satır (sütunlar: id, name, value).")
        elif selected_step == 'Transform':
            print("Transform Adımı: Çekilen veriler iş kurallarına göre dönüştürüldü. Eksik değerler dolduruldu, kategorik değişkenler kodlandı.")
            print("Dönüştürülmüş veri önizlemesi: İlk 5 satır (sütunlar: id, name_encoded, value_normalized).")
        elif selected_step == 'Report':
            print("Rapor durumu: Oluşturuldu ve \"raporlar\" dizinine kaydedildi.")
        else:
            print("Yukarıdaki DAG adımlarından birini seçerek detaylarını görüntüleyin.")

# Dropdown menüde bir değişiklik olduğunda fonksiyonu çağırın
dropdown.observe(on_dropdown_change, names='value')

Output()

HTML(value="<b style='color: black;'>DAG Bekliyor</b>")

IntProgress(value=0, bar_style='info', description='İlerleme:', max=3)

Output()

GridspecLayout(children=(HTML(value='<b>Task Adı</b>', layout=Layout(grid_area='widget001')), HTML(value='<b>D…

Button(description='DAG Çalıştır', style=ButtonStyle())

Button(description='Sonuçları İndir', disabled=True, style=ButtonStyle())

Button(description='Grafiği PNG Olarak Kaydet', disabled=True, style=ButtonStyle())

Dropdown(description='Adım Detayı:', options=('DAG Adımı Seçin', 'Extract', 'Transform', 'Report'), value='DAG…

In [7]:
import matplotlib.pyplot as plt

def create_summary_plot(save_png=False):
    task_names = []
    task_durations = []

    for i in range(1, 4):
        name = task_results_table[i, 0].value
        duration_str = task_results_table[i, 2].value

        # Ensure duration_str is a valid number before converting
        if duration_str != '-' and 'Çalışıyor' not in duration_str:
            task_names.append(name)
            task_durations.append(float(duration_str))
        else:
            # Assign 0 for incomplete tasks for plotting purposes
            task_names.append(name)
            task_durations.append(0.0)

    if not task_names:
        with output_area:
            print("No task data available to plot.")
        return

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(task_names, task_durations, color=['skyblue', 'lightcoral', 'lightgreen'])
    ax.set_xlabel('Task Name')
    ax.set_ylabel('Duration (seconds)')
    ax.set_title('DAG Task Durations Summary')
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    for index, value in enumerate(task_durations):
        if value > 0:
            ax.text(index, value + 0.1, f'{value:.2f}', ha='center', va='bottom')

    plt.tight_layout()

    if save_png:
        plot_file_name = f"dag_summary_plot_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
        fig.savefig(plot_file_name)
        with output_area:
            print(f"\nGrafik kaydedildi: {plot_file_name}")
            display(FileLink(plot_file_name))

    with output_area:
        display(fig.canvas) # Display the plot in the output area
    plt.close(fig)

# Modify run_dag_ipy to call the plotting function after completion
def run_dag_ipy_with_plot(b):
    run_dag_ipy(b)
    create_summary_plot()

# Update the button's on_click event to use the new function
button.on_click(run_dag_ipy_with_plot)
